# Project Additional Materials — Monthly AutoML Models (XGB / RF / LGB)

- Student ID: 10841269  
- Course Code: DATA70132  
- Academic Year: 2024–25  

**Environment:** See `README` and `ERP_Environment_2025.yaml`.  
**Reproduction:** Run this notebook top to bottom in the same folder as `All_UKonly_monthly_cleaned.nc`.  

This notebook trains AutoML models (XGBoost, Random Forest, LightGBM via FLAML) on monthly data and saves the fitted models and logs.

## Step 0 — Import required libraries
Load the Python packages required for monthly AutoML model training.  
(Dependencies are listed in the `README` and `ERP_Environment_2025.yaml`.)

In [1]:
import pandas as pd
import xarray as xr
import numpy as np
import random
import warnings
import pickle
from flaml import AutoML
from sklearn.metrics import r2_score, mean_squared_error

warnings.filterwarnings('ignore')

## Step 1 — Load monthly dataset & prepare training data


In [ ]:
# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)

# Load dataset
ds = xr.open_dataset("All_UKonly_monthly_cleaned.nc", decode_times=False)
df = ds.to_dataframe().dropna().reset_index()

# Time split
df_train = df[(df["year_month"] >= 200601) & (df["year_month"] <= 202012)]

# Features and target
features = ["TREFHT", "FLNS", "FSNS", "QBOT", "UBOT", "VBOT", "PRECT", "PRSN"]
target = "TREFMXAV_U"

X_train, y_train = df_train[features], df_train[target]


## Step 2 — Train XGBoost AutoML model with FLAML
Train the models and select the best one to save `xgb_monthly.pkl`.


In [ ]:
# XGBoost AutoML model
automl_xgb = AutoML()
automl_xgb.fit(
    X_train, y_train, # Training data
    task="regression", # Regression task
    estimator_list=["xgboost"], # Use only XGBoost
    time_budget=3600, # 1 hour
    metric="r2", # Optimize for R^2
    eval_method="cv", # Cross-validation
    n_jobs=-1, # Use all available cores
    log_file_name="xgb_monthly.log" # Log file name
)

# Print best model details
print("\n XGBoost Best Config:", automl_xgb.best_config)
print("Validation Loss:", automl_xgb.best_loss)

# Save the trained model
model_save_path = 'xgb_monthly.pkl'
with open(f"{model_save_path}", "wb") as f:
    pickle.dump(automl_xgb, f, pickle.HIGHEST_PROTOCOL)
print(f"Model for UK urban temperature prediction saved as {model_save_path}")


[flaml.automl.logger: 08-16 15:26:41] {1752} INFO - task = regression
[flaml.automl.logger: 08-16 15:26:41] {1763} INFO - Evaluation method: cv
[flaml.automl.logger: 08-16 15:26:41] {1862} INFO - Minimizing error metric: 1-r2
[flaml.automl.logger: 08-16 15:26:41] {1979} INFO - List of ML learners in AutoML Run: ['xgboost']
[flaml.automl.logger: 08-16 15:26:41] {2282} INFO - iteration 0, current learner xgboost
[flaml.automl.logger: 08-16 15:26:42] {2417} INFO - Estimated sufficient time budget=11834s. Estimated necessary time budget=12s.
[flaml.automl.logger: 08-16 15:26:42] {2466} INFO -  at 1.6s,	estimator xgboost's best error=0.4976,	best estimator xgboost's best error=0.4976
[flaml.automl.logger: 08-16 15:26:42] {2282} INFO - iteration 1, current learner xgboost
[flaml.automl.logger: 08-16 15:26:43] {2466} INFO -  at 2.8s,	estimator xgboost's best error=0.4976,	best estimator xgboost's best error=0.4976
[flaml.automl.logger: 08-16 15:26:43] {2282} INFO - iteration 2, current learne

## Step 3 — Train Random Forest AutoML model with FLAML
Train the models and select the best one to save `rf_monthly.pkl`.

In [ ]:
# Random Forest AutoML model
automl_rf = AutoML()
automl_rf.fit(
    X_train, y_train, # Training data
    task="regression", # Regression task
    estimator_list=["rf"], # Use only Random Forest
    time_budget=3600, # 1 hour
    metric="r2", # Optimize for R^2
    eval_method="cv", # Cross-validation
    n_jobs=-1, # Use all available cores
    log_file_name="rf_monthly.log" # Log file name
)

# Print best model details
print("\n Random Forest Best Config:", automl_rf.best_config)
print("Validation Loss:", automl_rf.best_loss)

# Save the trained model
model_save_path = 'rf_monthly.pkl'
with open(f"{model_save_path}", "wb") as f:
    pickle.dump(automl_rf, f, pickle.HIGHEST_PROTOCOL)
print(f"Model for UK urban temperature prediction saved as {model_save_path}")


[flaml.automl.logger: 08-16 16:25:24] {1752} INFO - task = regression
[flaml.automl.logger: 08-16 16:25:24] {1763} INFO - Evaluation method: cv
[flaml.automl.logger: 08-16 16:25:24] {1862} INFO - Minimizing error metric: 1-r2
[flaml.automl.logger: 08-16 16:25:24] {1979} INFO - List of ML learners in AutoML Run: ['rf']
[flaml.automl.logger: 08-16 16:25:24] {2282} INFO - iteration 0, current learner rf
[flaml.automl.logger: 08-16 16:25:33] {2417} INFO - Estimated sufficient time budget=91677s. Estimated necessary time budget=92s.
[flaml.automl.logger: 08-16 16:25:33] {2466} INFO -  at 9.7s,	estimator rf's best error=0.1241,	best estimator rf's best error=0.1241
[flaml.automl.logger: 08-16 16:25:33] {2282} INFO - iteration 1, current learner rf
[flaml.automl.logger: 08-16 16:25:43] {2466} INFO -  at 19.0s,	estimator rf's best error=0.0475,	best estimator rf's best error=0.0475
[flaml.automl.logger: 08-16 16:25:43] {2282} INFO - iteration 2, current learner rf
[flaml.automl.logger: 08-16 1

## Step 4 — Train LightGBM AutoML model with FLAML
Train the models and select the best one to save `lgb_monthly.pkl`.

In [ ]:
# LightGBM AutoML model
automl_lgb = AutoML()
automl_lgb.fit(
    X_train, y_train, # Training data
    task="regression", # Regression task
    estimator_list=["lgbm"], # Use only LightGBM
    time_budget=3600, # 1 hour
    metric="r2", # Optimize for R^2
    eval_method="cv", # Cross-validation
    n_jobs=-1, # Use all available cores
    log_file_name="lgb_monthly.log" # Log file name
)

# Print best model details
print("\n LightGBM Best Config:", automl_lgb.best_config)
print("Validation Loss:", automl_lgb.best_loss)

# Save the trained model
model_save_path = 'lgb_monthly.pkl'
with open(f"{model_save_path}", "wb") as f:
    pickle.dump(automl_lgb, f, pickle.HIGHEST_PROTOCOL)
print(f"Model for UK urban temperature prediction saved as {model_save_path}")

[flaml.automl.logger: 08-16 17:24:01] {1752} INFO - task = regression
[flaml.automl.logger: 08-16 17:24:01] {1763} INFO - Evaluation method: cv
[flaml.automl.logger: 08-16 17:24:01] {1862} INFO - Minimizing error metric: 1-r2
[flaml.automl.logger: 08-16 17:24:01] {1979} INFO - List of ML learners in AutoML Run: ['lgbm']
[flaml.automl.logger: 08-16 17:24:01] {2282} INFO - iteration 0, current learner lgbm
[flaml.automl.logger: 08-16 17:24:04] {2417} INFO - Estimated sufficient time budget=25700s. Estimated necessary time budget=26s.
[flaml.automl.logger: 08-16 17:24:04] {2466} INFO -  at 3.9s,	estimator lgbm's best error=0.4977,	best estimator lgbm's best error=0.4977
[flaml.automl.logger: 08-16 17:24:04] {2282} INFO - iteration 1, current learner lgbm
[flaml.automl.logger: 08-16 17:24:06] {2466} INFO -  at 6.0s,	estimator lgbm's best error=0.4977,	best estimator lgbm's best error=0.4977
[flaml.automl.logger: 08-16 17:24:06] {2282} INFO - iteration 2, current learner lgbm
[flaml.automl.